# Automatyzacja testów integracyjnych (Wariant A)

- **REST API** (`tests/integration/api/`, I1–I10): aplikacja FastAPI + moduły `utils`, `note_index`, `share`, `export`, `themes`, `plugins` + system plików,
- **MCP** (`tests/integration/mcp/`, I11–I17): serwer MCP → klient HTTP → REST API → system plików, przez endpoint `/mcp` i przez stdio.

Razem 99 przypadków testowych w 15 plikach.

## Czy testy spełniają wymagania

| Wymóg | Spełniony? | Jak |
|---|---|---|
| przygotowuje wymagany stan | tak | fixture'y w `tests/conftest.py`: `vault` (pusty katalog notatek w `tmp_path`), `make_note` (notatki zapisane na dysku), `auth` / `auth_client` / `logged_in_client` (włączone logowanie, sesja), `live_server` (prawdziwy serwer uvicorn na wolnym porcie), `plugin` (atrapa pluginu). Zmienne środowiskowe ustawiane są przed importem aplikacji. |
| sprawdza rezultat integracji | tak | każdy test sprawdza skutek po drugiej stronie integracji, a nie tylko kod HTTP: plik na dysku (`(vault / "a.md").read_text()`), wynik innego endpointu (wyszukiwarka, tagi i graf po zapisie, przeniesieniu i usunięciu), plik `.share-tokens.json`, zawartość archiwum ZIP, notatkę utworzoną narzędziem MCP. |
| zawiera jednoznaczne asercje | tak, z kilkoma wyjątkami | prawie wszędzie porównujemy dokładne wartości (`== 409`, `== ["a.md", "b.md"]`, `== {"alpha": 2, "beta": 1}`). Słabsze asercje opisujemy niżej w *Ograniczeniach*. Znane defekty mają osobne testy `xfail(strict=True)`: test sprawdza poprawne zachowanie, a jeśli ktoś naprawi błąd, `strict` zgłosi to jako niespodziewane przejście. |
| daje się uruchomić ponownie | tak | dwa przebiegi pod rząd dają ten sam wynik (komórka niżej). Sprawdziliśmy też każdy plik osobno i całość w odwrotnej kolejności: zawsze 95 passed + 4 xfailed. Testy nie zależą od siebie, bo autouse fixture `_reset_caches` czyści indeks notatek i cache skanowania przed i po każdym teście, a każdy test dostaje własny katalog. |
| sprząta dane / izolowane środowisko | tak | dane testów lądują tylko w `tmp_path` (katalog tymczasowy pytesta, osobny dla każdego testu). Zmiany konfiguracji robimy przez `monkeypatch`, który cofa je po teście. `live_server` jest zatrzymywany po teście. Katalog startowy `/tmp/nd-tests-*` (ustawiany jako `NOTES_DIR` / `PLUGINS_DIR` przed importem aplikacji) usuwa `pytest_unconfigure` w `tests/conftest.py`. W repozytorium ani w `/tmp` nic nie zostaje (komórka niżej).

## Użyte narzędzia

- **pytest 9.2** (Python 3.12) jako runner, fixture'y i markery `integration` / `mcp`,
- **FastAPI `TestClient`** (httpx 0.27) do testów REST: wywołuje aplikację w tym samym procesie, bez sieci,
- **uvicorn 0.54** uruchamiany w wątku (`live_server`): prawdziwy serwer HTTP na `127.0.0.1` i wolnym porcie, bo klient MCP (`mcp_server/client.py`) używa `urllib` i nie umie rozmawiać z `TestClient`,
- **httpx** do wysyłania surowych żądań do `/mcp` (nagłówki, błędny JSON, inne metody HTTP),
- **subprocess** do uruchomienia `python -m mcp_server` (transport stdio) i do testu decyzji o montowaniu `/mcp` przy starcie aplikacji,
- **monkeypatch** i **tmp_path** z pytesta do izolacji.

## Instrukcja uruchomienia

Z katalogu głównego repozytorium:

```bash
uv sync --extra dev                      # albo: pip install -e ".[dev]"
uv run pytest tests/integration -m integration -rxX   # wszystkie testy integracyjne
uv run pytest tests/integration -m mcp                # tylko MCP
uv run pytest tests/integration/api                   # tylko REST API
```

Nie trzeba niczego uruchamiać ręcznie (serwera, bazy, Dockera). Testy same ustawiają zmienne środowiskowe, tworzą katalogi tymczasowe i startują serwer. `-rxX` wypisuje powody testów `xfail`.

## Uruchomienie i wynik testu

Komórka uruchamia testy dwa razy pod rząd i porównuje stan repozytorium (`git status --ignored`) oraz katalogi `/tmp/nd-tests-*` przed i po.

In [1]:
import subprocess, sys, tempfile, glob
from pathlib import Path

ROOT = Path("../..").resolve()
WYNIKI = Path(tempfile.mkdtemp(prefix="wyniki-integracja-"))

def stan_repo():
    """Pliki zmienione / nowe / ignorowane w repozytorium (do sprawdzenia, czy testy coś zostawiły)."""
    return subprocess.run(["git", "status", "--short", "--ignored"], cwd=ROOT,
                          capture_output=True, text=True).stdout

przed = stan_repo()
katalogi_przed = set(glob.glob(f"{tempfile.gettempdir()}/nd-tests-*"))
for nr in (1, 2):  # dwa przebiegi pod rząd: test ma się dać uruchomić ponownie z tym samym wynikiem
    out = subprocess.run(
        [sys.executable, "-m", "pytest", "tests/integration", "-m", "integration", "-q", "-rxX",
         "-p", "no:cacheprovider", "-W", "ignore::DeprecationWarning",
         f"--junitxml={WYNIKI / f'przebieg{nr}.xml'}"],
        cwd=ROOT, capture_output=True, text=True,
    )
    print(f"=== przebieg {nr} (kod wyjścia {out.returncode})")
    print("\n".join(l for l in out.stdout.splitlines() if l.startswith(("XFAIL", "FAILED", "ERROR")) or " in " in l))

print("\nrepozytorium bez zmian po testach:", przed == stan_repo())
print("nowe katalogi /tmp/nd-tests-* po testach:", len(set(glob.glob(f"{tempfile.gettempdir()}/nd-tests-*")) - katalogi_przed))

=== przebieg 1 (kod wyjścia 0)
XFAIL tests/integration/api/test_api_folders.py::test_delete_missing_folder_is_404 - DEFECT: deleting a folder that does not exist returns 500 (delete_folder() -> False is mapped to 500) instead of 404
XFAIL tests/integration/api/test_api_notes.py::test_traversal_write_is_rejected_as_client_error - DEFECT: a path traversal write is refused, but answered with 500 (server error) instead of 400/403 (client error)
XFAIL tests/integration/mcp/test_mcp_note_tools.py::test_failures_set_is_error - DEFECT: REST errors (404/409) reach the MCP client as plain text, the tools/call result is not marked isError
XFAIL tests/integration/mcp/test_mcp_stdio.py::test_wrong_api_key_sets_is_error - DEFECT: a 401 from the backend reaches the stdio client as plain text, the tools/call result is not marked isError
95 passed, 4 xfailed, 1 warning in 15.59s
=== przebieg 2 (kod wyjścia 0)
XFAIL tests/integration/api/test_api_folders.py::test_delete_missing_folder_is_404 - DEFECT: d

In [2]:
import xml.etree.ElementTree as ET
from collections import Counter

wynik = Counter()
for case in ET.parse(WYNIKI / "przebieg2.xml").iter("testcase"):
    plik = case.get("classname").rsplit(".", 1)[-1] + ".py"
    stan = "xfail" if case.find("skipped") is not None else \
           "fail" if case.find("failure") is not None or case.find("error") is not None else "pass"
    wynik[(plik, stan)] += 1

pliki = sorted({p for p, _ in wynik})
print(f"{'plik':34} {'pass':>5} {'xfail':>6} {'fail':>5}")
for p in pliki:
    print(f"{p:34} {wynik[(p,'pass')]:>5} {wynik[(p,'xfail')]:>6} {wynik[(p,'fail')]:>5}")
print(f"{'RAZEM':34} {sum(v for (p,s),v in wynik.items() if s=='pass'):>5} "
      f"{sum(v for (p,s),v in wynik.items() if s=='xfail'):>6} {sum(v for (p,s),v in wynik.items() if s=='fail'):>5}")

plik                                pass  xfail  fail
test_api_auth.py                       9      0     0
test_api_export.py                     6      0     0
test_api_folders.py                    6      1     0
test_api_media.py                      6      0     0
test_api_notes.py                      5      1     0
test_api_search_graph.py               5      0     0
test_api_sharing.py                   11      0     0
test_api_system_plugins.py             3      0     0
test_api_templates.py                  4      0     0
test_mcp_auth.py                       9      0     0
test_mcp_note_tools.py                 9      1     0
test_mcp_read_tools.py                 5      0     0
test_mcp_session.py                    4      0     0
test_mcp_stdio.py                      4      1     0
test_mcp_transport.py                  9      0     0
RAZEM                                 95      4     0


## Elementy rzeczywiste i atrapy

Aplikacja nie korzysta z żadnych zewnętrznych usług (baza danych to pliki Markdown na dysku), więc prawie cały łańcuch w testach jest prawdziwy.

| Element | Rzeczywisty / atrapa | Uwagi |
|---|---|---|
| aplikacja FastAPI (`backend/main.py`): routing, walidacja, auth (sesja, klucz API), obsługa błędów | **rzeczywisty** | |
| moduły `utils`, `note_index`, `share`, `export`, `themes`, locales | **rzeczywiste** | |
| system plików (notatki, media, `.share-tokens.json`, `_templates`) | **rzeczywisty**, ale w katalogu tymczasowym | `NOTES_DIR` wskazuje na `tmp_path`, nie na `data/` |
| serwer MCP (`mcp_server.server`), klient HTTP MCP (`mcp_server.client`), endpoint `/mcp` | **rzeczywiste** | |
| serwer HTTP uvicorn i połączenia TCP (testy MCP) | **rzeczywiste** | na `127.0.0.1`, wolny port |
| proces `python -m mcp_server` (stdio) | **rzeczywisty** | osobny proces |
| warstwa transportu w testach REST | **zastąpiona** | `TestClient` wywołuje aplikację w tym samym procesie (ASGI), bez prawdziwej sieci |
| konfiguracja (`config.yaml`) | **częściowo zastąpiona** | zmienne środowiskowe przy imporcie, `monkeypatch` na `main.config` (logowanie, wyszukiwarka). Hasło: hash bcrypt tworzony w teście |
| pluginy | **atrapa** | `PLUGINS_DIR` pusty, w testach hooków wstrzykiwany jest sztuczny plugin `Upper` (zamienia treść na wielkie litery), plik konfiguracji pluginów przekierowany do `tmp_path` |
| limity i ustawienia (`UPLOAD_MAX_IMAGE_MB = 0`, `SHARE_PUBLIC_ORIGIN`) | **podmienione** `monkeypatch` | żeby wywołać gałąź „za duży plik” i sprawdzić adres URL linku bez prawdziwej domeny |
| obrazy PNG | **atrapa** | bajty z poprawnym nagłówkiem PNG i sztuczną treścią |
| `test_origin_allowlist` | **atrapa** | osobna, minimalna aplikacja z `build_mcp_router` i funkcjami-zaślepkami zamiast klucza API i adresu loopback |
| `test_loopback_url` | **atrapa** | żądanie HTTP zastąpione `SimpleNamespace` |
| niedostępny backend (`test_backend_down`) | **symulowany** | adres na zamkniętym porcie |
| daty modyfikacji notatek (`get_recent_notes`) | **podmienione** | `os.utime` ustawia mtime w przeszłości. Zegar systemowy nie jest mockowany |
| rate limiter (tryb demo) | **wyłączony** | `DEMO_MODE` usuwane ze środowiska |